# **Entrenamiento final y submission**

Este notebook no reimplementa el pipeline: es un orquestador delgado sobre
`scripts/train.py` y `scripts/predict.py`, que a su vez reutilizan
`src/features/build_features.py`, `src/models/train_model.py` y
`src/models/predict.py` (las mismas piezas usadas en `03-model.ipynb`).

El modelo entrenado es el ganador documentado en
[`docs/model_selection.md`](../docs/model_selection.md) (actualmente
**CatBoost**, criterio: WAPE % más bajo).

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()

def run(script: str, *args: str) -> None:
    """Corre un script de scripts/ como subproceso, con stdout/stderr en vivo."""
    cmd = [sys.executable, str(PROJECT_ROOT / 'scripts' / script), *args]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## 1. Entrenar el modelo ganador

`scripts/train.py`:
1. Carga `data/3.final/train_final.csv` y aplica `build_temporal_features`.
2. Separa un hold-out temporal del 12% más reciente para encontrar el número
   de rondas óptimo con early stopping (no un valor fijo sin validar).
3. Reentrena con el 100% de los datos usando ese número de rondas.
4. Guarda el modelo en `models/best_model.pkl`.

In [ ]:
run('train.py')

## 2. Generar la submission

`scripts/predict.py` carga `models/best_model.pkl`, reconstruye `df_all`
(train + test concatenados) y corre la predicción recursiva (walk-forward,
`recursive_predict` en `src/models/predict.py`) para generar
`data/3.final/submission_best.csv`.

In [ ]:
run('predict.py')

## 3. Verificación rápida de la submission

In [ ]:
import pandas as pd

submission = pd.read_csv(PROJECT_ROOT / 'data' / '3.final' / 'submission_best.csv')
print(submission.shape)
print(submission['sales'].describe())
submission.head()